In [ ]:
# %%
#import packages
from flask import Flask, render_template, jsonify
import json
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from collections import Counter
import io
from datetime import datetime, timedelta
from IPython.display import display, HTML
from dotenv import load_dotenv

load_dotenv()

#name app and invent a 'SECRET_KEY' that is not really used for anything
app = Flask(__name__)
app.config['SECRET_KEY'] = ''


#tell flask to read home page
@app.route('/')
def index(): 
    return render_template('/dashboard_public.html')



# Path to local parquet file
PARQUET_PATH = ""


# Simple in-memory cache
_cache = {"data": None, "fields": None, "fetched_at": None}
CACHE_TTL = timedelta(minutes=15)

def cache_is_fresh():
    return (
        _cache["fetched_at"] is not None and
        datetime.now() - _cache["fetched_at"] < CACHE_TTL
    )

def get_all_features():
    gdf = gpd.read_parquet(PARQUET_PATH)

    # Ensure CRS is WGS84
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    geojson = json.loads(gdf.to_json())
    print(f"Total features loaded: {len(geojson['features'])}")
    return geojson

def get_numeric_fields():
    gdf = gpd.read_parquet(PARQUET_PATH)

    skip_names = {"objectid", "oid", "fid", "shape_area", "shape_length",
                  "shape__area", "shape__length"}

    numeric_dtypes = ["int8", "int16", "int32", "int64",
                      "float16", "float32", "float64"]

    return [
        col for col in gdf.columns
        if gdf[col].dtype.name in numeric_dtypes
        and col.lower() not in skip_names
    ]


@app.route('/hexdata')
def api_hexdata():
    try:
        if not cache_is_fresh():
            _cache["data"] = get_all_features()
            _cache["fetched_at"] = datetime.now()
        return jsonify(_cache["data"])
    except FileNotFoundError:
        return jsonify({'error': f'Parquet file not found: {PARQUET_PATH}'}), 500
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/fields')
def api_fields():
    try:
        if not _cache["fields"]:
            _cache["fields"] = get_numeric_fields()
        return jsonify(_cache["fields"])
    except FileNotFoundError:
        return jsonify({'error': f'Parquet file not found: {PARQUET_PATH}'}), 500
    except Exception as e:
        return jsonify({'error': str(e)}), 500
    
@app.route('/debug')
def debug():
    try:
        import os
        exists = os.path.exists(PARQUET_PATH)
        cwd = os.getcwd()
        files = os.listdir(cwd)
        return jsonify({
            "parquet_path": PARQUET_PATH,
            "file_exists": exists,
            "cwd": cwd,
            "files_in_cwd": files
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

## check count of features fetched
fc = get_all_features()
print(len(fc["features"]))

#tell flask run app
if __name__ == '__main__':
    app.run(host='0.0.0.0')

Total features loaded: 10037
10037
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.216:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Jul/2026 15:12:19] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Jul/2026 15:12:19] "GET /static/stylesheet2.css HTTP/1.1" 304 -
127.0.0.1 - - [13/Jul/2026 15:12:20] "GET /static/PlanRVA_White.png HTTP/1.1" 304 -
127.0.0.1 - - [13/Jul/2026 15:12:20] "GET /fields HTTP/1.1" 200 -
127.0.0.1 - - [13/Jul/2026 15:12:21] "GET /hexdata HTTP/1.1" 200 -


Total features loaded: 10037
